In [ ]:
import sqlalchemy
import pandas as pd
import numpy as np
from google.colab import userdata
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.ticker as mtick

In [ ]:
DB_USER = userdata.get('DB_USER_2')
DB_PASSWORD = userdata.get('DB_PASSWORD_2')
DB_HOST = userdata.get('DB_HOST_2')
DB_NAME = userdata.get('DB_NAME_2')
DB_PORT = "5432"

In [ ]:
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = sqlalchemy.create_engine(connection_string)

In [ ]:
pd.set_option('display.max_columns', None)

In [ ]:
all_data_so_far = """
SELECT *
FROM public.abmf_table_2024
ORDER BY time DESC
LIMIT 100;
"""

all_data_df = pd.read_sql(all_data_so_far, engine)

In [ ]:
all_data_df.head()

In [ ]:
len(all_data_df.columns)

In [ ]:
earliest_data = """
SELECT *
FROM public.abmf_table_2024
ORDER BY time ASC
LIMIT 100;
"""

earliest_df = pd.read_sql(earliest_data, engine)

In [ ]:
earliest_df.sample(5)

09/06

In [ ]:
lse_abmf_data_query = """
SELECT
    "time",
    site,
    month,
    day,
    year,

    -- Power source (critical for grid vs generator detection)
    powersource,
    multi_powersource,

    -- Voltage (outage + power quality)
    vnavg_v,
    vlavg_v,
    freq_hz,

    -- Current and power factor
    iavg_a,
    pf,

    -- Active power (load sizing)
    p_kw,
    pa_kw,
    pb_kw,
    pc_kw,

    -- Grid vs generator power and energy (core for emissions)
    grid_kw,
    gen1_kw,
    gen2_kw,
    grid_kwh,
    gen1_kwh,
    gen2_kwh,
    system_kw,
    system_kwh,

    -- Operational hours (outage duration proxy)
    grid_hours,
    gen1_hours,
    gen2_hours,
    total_hours,

    -- Fuel consumption (emissions estimation)
    gen1_litres_model,
    gen2_litres_model,
    diesel_estimated_1,
    diesel_estimated_2,

    -- Cost (direct financial cost dimension)
    cost_gen,
    cost_grid,
    total_cost,
    diesel_prices,
    grid_tariff,
    daily_price_of_used,

    -- Carbon
    carbon_footprint

FROM public.abmf_table_2024

WHERE site IN ('Head Office', 'Ikeja')
AND "time" >= '2025-01-01 00:00:00'
AND "time" < '2025-04-01 00:00:00'

ORDER BY site, "time" ASC;
"""

lse_abmf_df = pd.read_sql(lse_abmf_data_query, engine)

In [ ]:
lse_abmf_df.head()

In [ ]:
df = lse_abmf_df.copy()

In [ ]:
df.columns

In [ ]:
df.powersource.unique()

In [ ]:
df.powersource.value_counts()

In [ ]:
df.multi_powersource.unique()

In [ ]:
target_sources = ['Grid', 'Gen1', 'Gen2']

filtered_df = df[df['multi_powersource'].isin(target_sources)]

In [ ]:
filtered_df[filtered_df['powersource'] == 'probably Grid']

In [ ]:
df[df['powersource'] == 'probably Grid']

In [ ]:
# 1. Ensure datetime format
df['time'] = pd.to_datetime(df['time'])

# 2. MERGE DUPLICATES FIRST (Choose .mean(), .sum(), or .last() based on your data)
df_cleaned = df.groupby(['site', 'time']).mean().reset_index()

# 3. Now set the index
df_cleaned = df_cleaned.set_index('time')

# 4. Run the resample step exactly like before
df_continuous = (
    df_cleaned.groupby('site')
    .resample('1min')
    .ffill()
    .drop(columns='site')
    .reset_index()
)

In [ ]:
len(lse_abmf_df)

In [ ]:
lse_abmf_df.columns

In [ ]:
'site', 'powersource', 'gen1_kw', 'gen2_kw', 'gen1_kwh', 'gen2_kwh', 'gen1_hours', 'gen2_hours', 'diesel_prices', 'gen1_litres_model', 'gen2_litres_model'

In [ ]:
import pandas as pd
import numpy as np

# --- Step 1: Prep and Sort Data ---
# Ensure your time column is parsed as a datetime object
df['time'] = pd.to_datetime(df['time'])
df = df.sort_values(['site', 'time'])

# --- Step 1.5: Remove Duplicate Timestamps per Site ---
# This fixes the ValueError by keeping only the first entry if a minute is logged twice
df = df.drop_duplicates(subset=['site', 'time'], keep='first')

# --- Step 2: Expose Missing Intervals per Site ---
def pad_and_detect_outages(group):
    # Resample to strict 1-minute intervals to expose gaps
    resampled = group.set_index('time').resample('1min').asfreq()

    # If 'site' is NaN, it means Python injected this row because it was a missing minute
    resampled['is_outage'] = resampled['site'].isna().astype(int)

    # Restore the site name for the newly injected rows
    resampled['site'] = group['site'].iloc[0]

    return resampled

# Apply the padding function across both sites
df_padded = df.groupby('site', group_keys=False).apply(pad_and_detect_outages)

# --- Step 3: Define the Aggregation Rules ---
avg_min_max_cols = [
    'vnavg_v', 'vlavg_v', 'freq_hz', 'iavg_a', 'pf', 'p_kw', 'pa_kw',
    'pb_kw', 'pc_kw', 'grid_kw', 'gen1_kw', 'gen2_kw', 'system_kw'
]

sum_cols = [
    'grid_kwh', 'gen1_kwh', 'gen2_kwh', 'system_kwh', 'grid_hours',
    'gen1_hours', 'gen2_hours', 'total_hours', 'gen1_litres_model',
    'gen2_litres_model', 'diesel_estimated_1', 'diesel_estimated_2',
    'cost_gen', 'cost_grid', 'total_cost', 'daily_price_of_used',
    'carbon_footprint', 'is_outage'
]

# Build the aggregation dictionary dynamically
agg_dict = {}
for col in avg_min_max_cols:
    agg_dict[col] = ['mean', 'min', 'max']
for col in sum_cols:
    agg_dict[col] = 'sum'

# For categorical metadata or fixed pricing, we grab the first available record
for col in ['powersource', 'multi_powersource', 'diesel_prices', 'grid_tariff', 'month', 'day', 'year']:
    agg_dict[col] = 'first'

# --- Step 4: Aggregate to 10-Minute Windows ---
# Group by site and resample the datetime index to 10 minutes
df_10min = df_padded.groupby('site').resample('10min').agg(agg_dict)

# --- Step 5: Clean up Column Schemas ---
# Flatten the MultiIndex columns created by the mean/min/max transformations
flat_cols = []
for col, stat in df_10min.columns:
    if stat in ['mean', 'min', 'max']:
        flat_cols.append(f"{col}_{stat}")
    elif col == 'is_outage':
        flat_cols.append('outage_minutes')  # Renamed for clarity
    else:
        flat_cols.append(col)

df_10min.columns = flat_cols
df_10min = df_10min.reset_index()

In [ ]:
df_10min['site'] = df_10min['site'].replace({'Head Office': 'HQ', 'Ikeja': 'Branch_1'})

In [ ]:
df_10min.to_csv('smarterise_commercial_client_data.csv')

In [ ]:
import shutil

source_path = '/content/smarterise_commercial_client_data.csv'
destination_path = '/content/drive/MyDrive/LSE Project/smarterise_commercial_client_data.csv'

shutil.move(source_path, destination_path)

Make the timestamp continous and fi

Head Office --> HQ

Ikeja --> BRANCH1

In [ ]:
all_litres_data = """
SELECT starting_litres, reported_liter
FROM public.abmf_table_2024
ORDER BY time DESC
"""

all_litres_df = pd.read_sql(all_litres_data, engine)

In [ ]:
all_litres_df.describe()

In [ ]:
# 1. Replace all 0s with NaN
df_cleaned = all_litres_df.replace(0, np.nan)

# 2. Count non-zero (non-NaN) values per column
non_zero_counts = df_cleaned.count()

# 3. Get the percentage of non-zero data per column
non_zero_percentage = df_cleaned.count() / len(all_litres_df) * 100

In [ ]:
non_zero_percentage

In [ ]:
all_litres_df[all_litres_df['reported_liter'] != 0]

In [ ]:
# For ifeanyi

all_litres_data = """
SELECT
    site,
    powersource,
    gen1_kw,
    gen2_kw,
    gen1_kwh,
    gen2_kwh,
    gen1_hours,
    gen2_hours,
    reported_liter,
    diesel_prices,
    gen1_litres_model,
    gen2_litres_model
FROM public.abmf_table_2024
WHERE powersource IN ('Gen1', 'Gen2')
ORDER BY time DESC
"""

all_litres_df = pd.read_sql(all_litres_data, engine)

In [ ]:
all_litres_df.to_csv('all_litres_data.csv', index=False)

In [ ]:
import shutil

source_path = '/content/all_litres_data.csv'
destination_path = '/content/drive/MyDrive/Diesel Consumption Modelling/all_litres_data.csv'

shutil.move(source_path, destination_path)

In [ ]:
updated_df = all_litres_df[all_litres_df['reported_liter'] != 0]

In [ ]:
updated_df.groupby('site')['reported_liter'].describe()